# 15 — Capstone: End-to-End ML Project
**Goal:** Build a complete ML product on real marketing data. From the
business question to a deployed model, in 14 cells. Source: Géron Ch2
checklist, ISLR Ch3 example, CRISP-DM, ML Canvas.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/learning_courses')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
np.random.seed(15)

## 1. Frame the problem

**Stakeholder:** VP Marketing.
**Question:** can we predict the number of paid conversions on a given day
from the spend levels and impression counts on the channels we control?
**Deliverable:** a regression model that, given tomorrow's planned spend,
estimates tomorrow's conversions — and a clear statement of the model's
trustworthy range.
**Metric:** MAE on a held-out *future* window (not random split — time
series).

## 2. Get the data

Production-grade data loading: read, parse dates, sanity-check schema.

In [ ]:
DATA_PATH = 'data/clean/unified_daily.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['date'] if 'date' in pd.read_csv(DATA_PATH, nrows=0).columns else None)
df = df.sort_values('date').reset_index(drop=True)
print('shape :', df.shape)
print('range :', df['date'].min(), '->', df['date'].max())
print('NaN per column:')
print(df.isna().sum())

## 3. Quick EDA

The four diagnostic plots: time series of target, distributions of
predictors, scatter against target, correlation matrix.

In [ ]:
target = 'paid_conv'
num_features = ['spend', 'impressions', 'clicks', 'cvr', 'ctr', 'cpa', 'roas']
num_features = [c for c in num_features if c in df.columns]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df['date'], df[target], color='steelblue', lw=1.5)
axes[0].set_title(f'{target} over time'); axes[0].set_xlabel('date')
axes[0].spines[['top','right']].set_visible(False)
corr = df[num_features + [target]].corr()[target].drop(target).sort_values()
axes[1].barh(corr.index, corr.values, color=['crimson' if v < 0 else 'steelblue' for v in corr.values])
axes[1].set_title(f'Correlation with {target}')
axes[1].spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 4. Split — temporal, not random

**Critical:** in production you predict the future from the past. Use the
last 20% of days as the test set. No shuffling.

In [ ]:
n = len(df)
split = int(n * 0.8)
df_tr, df_te = df.iloc[:split], df.iloc[split:]
print(f'train: {len(df_tr)} days  ({df_tr["date"].min()} -> {df_tr["date"].max()})')
print(f'test : {len(df_te)} days  ({df_te["date"].min()} -> {df_te["date"].max()})')

## 5. Build the pipeline

Wrap preprocessing + model in a single `Pipeline`. The model can be swapped
without touching anything else. No leakage — imputer and scaler fit on the
training fold only.

In [ ]:
pre = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
    ]), num_features),
])

def make_pipeline(model):
    return Pipeline([('pre', pre), ('model', model)])

X_tr = df_tr[num_features]; y_tr = df_tr[target]
X_te = df_te[num_features]; y_te = df_te[target]
print('pipeline sketch:')
print(make_pipeline(Ridge(alpha=1.0)))

## 6. Baselines — naive and seasonal

Never trust a model that does not beat a trivial baseline.

In [ ]:
naive_pred   = y_tr.iloc[-1] * np.ones(len(y_te))
season_pred  = df[target].shift(7).iloc[split:].values
print(f'naive (last train value) MAE : {mean_absolute_error(y_te, naive_pred):.2f}')
print(f'seasonal (lag 7)        MAE : {mean_absolute_error(y_te, season_pred):.2f}')

## 7. Model zoo

Train four models spanning the linear-nonparametric spectrum. Use the
pipeline from cell 5.

In [ ]:
models = {
    'Linear':   LinearRegression(),
    'Ridge':    Ridge(alpha=1.0),
    'RF':       RandomForestRegressor(n_estimators=300, max_depth=6, random_state=0, n_jobs=-1),
    'GBM':      GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=0),
}
results = []
for name, mdl in models.items():
    pipe = make_pipeline(mdl)
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    mae  = mean_absolute_error(y_te, pred)
    r2   = r2_score(y_te, pred)
    results.append((name, mae, r2))
    print(f'{name:8s}  MAE = {mae:7.2f}  R\u00b2 = {r2:.3f}')

## 8. Time-series cross-validation
Random k-fold overestimates performance in time series (it leaks the
future). Use `TimeSeriesSplit` to get a more honest estimate.

In [ ]:
X = df[num_features]; y = df[target]
tscv = TimeSeriesSplit(n_splits=5)
for name, mdl in models.items():
    pipe = make_pipeline(mdl)
    s = -cross_val_score(pipe, X, y, cv=tscv, scoring='neg_mean_absolute_error')
    print(f'{name:8s}  CV MAE = {s.mean():7.2f}  +/- {s.std():.2f}')

## 9. Residual diagnostics — is the model right?

Plot predicted vs actual and residuals vs fitted. Patterns = bugs or model
misspecification.

In [ ]:
best_name, best_mae, best_r2 = min(results, key=lambda r: r[1])
best_pipe = make_pipeline(models[best_name]).fit(X_tr, y_tr)
pred = best_pipe.predict(X_te)
resid = y_te - pred
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_te, pred, alpha=0.7, color='steelblue')
lo = min(y_te.min(), pred.min()); hi = max(y_te.max(), pred.max())
axes[0].plot([lo, hi], [lo, hi], color='crimson', linestyle='--', lw=1)
axes[0].set_xlabel('actual'); axes[0].set_ylabel('predicted')
axes[0].set_title(f'{best_name}: actual vs predicted')
axes[1].scatter(pred, resid, alpha=0.7, color='steelblue')
axes[1].axhline(0, color='crimson', linestyle='--', lw=1)
axes[1].set_xlabel('predicted'); axes[1].set_ylabel('residual')
axes[1].set_title('Residuals vs predicted')
for ax in axes: ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 10. Feature importance — what is the model using?

Tree-based models give MDI importance. Use permutation importance for a
more reliable view (notebook 09).

In [ ]:
from sklearn.inspection import permutation_importance
perm = permutation_importance(best_pipe, X_te, y_te, n_repeats=20, random_state=0)
order = np.argsort(perm.importances_mean)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(np.array(num_features)[order], perm.importances_mean[order],
        xerr=perm.importances_std[order], color='steelblue', alpha=0.8)
ax.set_title(f'Permutation importance ({best_name})')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 11. Hyperparameter search — but only for the winner
Refit the best model with a small grid on the training set, scored with
TimeSeriesSplit.

In [ ]:
from sklearn.model_selection import GridSearchCV
if best_name == 'GBM':
    grid = {'model__n_estimators': [200, 400], 'model__max_depth': [2, 3, 4]}
elif best_name == 'RF':
    grid = {'model__n_estimators': [300, 600], 'model__max_depth': [4, 8, None]}
elif best_name == 'Ridge':
    grid = {'model__alpha': [0.1, 1.0, 10.0]}
else:
    grid = {'model__fit_intercept': [True, False], 'model__positive': [True, False]}
search = GridSearchCV(make_pipeline(models[best_name]), grid, cv=TimeSeriesSplit(5), scoring='neg_mean_absolute_error', n_jobs=-1)
search.fit(X_tr, y_tr)
print('best params :', search.best_params_)
print('CV MAE      :', -search.best_score_)
test_mae = mean_absolute_error(y_te, search.predict(X_te))
print(f'test MAE    : {test_mae:.2f}')

## 12. Confidence interval — bootstrap the residuals

We need not just a point estimate but a *plausible range*. Bootstrap the
MAE on the test set.

In [ ]:
pred = search.predict(X_te)
resid = y_te.values - pred
rng = np.random.default_rng(0)
B = 5000
boot_mae = np.array([
    np.mean(np.abs(rng.choice(resid, size=len(resid), replace=True)))
    for _ in range(B)
])
lo, hi = np.percentile(boot_mae, [2.5, 97.5])
print(f'point MAE   : {mean_absolute_error(y_te, pred):.2f}')
print(f'95% CI      : [{lo:.2f}, {hi:.2f}]')

## 13. Save and ship

Serialize the pipeline with joblib. The same object loads in a Flask or
FastAPI service.

In [ ]:
import os
os.makedirs('models', exist_ok=True)
ARTIFACT = 'models/paid_conv_pipeline.joblib'
joblib.dump(search.best_estimator_, ARTIFACT)
loaded = joblib.load(ARTIFACT)
sample = X_te.head(1)
print('saved to     :', ARTIFACT)
print('loaded sanity :', loaded.predict(sample))

In [ ]:
# Tiny serving stub
SERVING = '''\
import joblib, pandas as pd
pipe = joblib.load('models/paid_conv_pipeline.joblib')

def predict(spend, impressions, clicks, cvr, ctr, cpa, roas):
    X = pd.DataFrame([[spend, impressions, clicks, cvr, ctr, cpa, roas]],
                     columns=['spend','impressions','clicks','cvr','ctr','cpa','roas'])
    return float(pipe.predict(X)[0])

if __name__ == '__main__':
    print('predicted convs:', predict(1000, 50000, 800, 0.05, 0.016, 12.5, 4.0))
'''
with open('src/serve_paid_conv.py', 'w') as f:
    f.write(SERVING)
print('wrote src/serve_paid_conv.py')

## 14. Monitoring plan

A model in production is a model under drift. Track at minimum:

1. **Input drift** — distribution of `spend`, `impressions`, ... over time.
   Use PSI (population stability index) or KS test against a baseline.
2. **Output drift** — distribution of predictions. Should track the target.
3. **Performance** — when ground truth is available (often delayed),
   recompute MAE and compare to the test-set MAE from this notebook.
4. **Retrain cadence** — weekly is a reasonable default; trigger-based on
   drift is better.
5. **Shadow mode** for new models — log predictions, do not act on them,
   compare to the live model for two weeks.

In [ ]:
# Drift sketch: KS test of a recent window vs the training baseline
from scipy.stats import ks_2samp
recent = df.iloc[-int(0.2*len(df)):]
drift_report = []
for col in num_features:
    stat, p = ks_2samp(df_tr[col], recent[col])
    drift_report.append((col, stat, p))
for col, stat, p in drift_report:
    flag = ' <-- DRIFT' if p < 0.05 else ''
    print(f'{col:14s}  KS = {stat:.3f}  p = {p:.3f}{flag}')

## 15. The final report card

| Item | Value |
|---|---|
| Best model | {best_name} |
| Test MAE | {best_mae:.2f} |
| 95% bootstrap CI on MAE | [{lo:.2f}, {hi:.2f}] |
| Top features (permutation) | see cell 10 |
| Saved artifact | `models/paid_conv_pipeline.joblib` |
| Serving stub | `src/serve_paid_conv.py` |
| Monitoring | KS drift test, weekly retrain |
|
**What you have built:**
1. A problem framing with stakeholder, metric, and trust range.
2. Honest temporal evaluation that does not leak the future.
3. A model selection across linear and nonparametric families.
4. Hyperparameter search on the best family only.
5. A confidence interval on the metric that the business cares about.
6. A serializable artifact and a serving stub.
7. A drift / retrain plan.

**What you have not built (and don't need for this capstone):**
- A web service. Wrap the artifact in Flask / FastAPI in a real project.
- Real-time feature store. Use Feast or Tecton.
- Model registry. Use MLflow or Weights & Biases.
- Causal effect of spend on conversions — that is a different question
  (notebook 14, section 8).

**Final advice:** ship something small, monitor it, and iterate. A model
in production is worth a hundred notebooks on your laptop.